**CI twin of `ch07-overfitting-regularization.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
from lib.data import load_csv
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import pandas as pd

homes = load_csv("california-housing-sample")
# Powers of big numbers explode (12^15 ≈ 10^16), so rescale income to 0–1.
# One honest line of preprocessing; the proper toolkit arrives in Ch16.
homes["x"] = homes["MedInc"] / 12.0

train = homes.sample(n=30, random_state=3)
test = homes.drop(train.index)
Xtr, ytr = train[["x"]], train["MedHouseVal"]
Xte, yte = test[["x"]], test["MedHouseVal"]

grid = pd.DataFrame({"x": [i / 200 for i in range(1, 201)]})
fig, ax = plt.subplots(figsize=(5.5, 3.4))
ax.scatter(Xtr["x"], ytr, s=18, zorder=3, label="30 training houses")
for degree in (1, 4, 15):
    pf = PolynomialFeatures(degree=degree, include_bias=False)
    m = LinearRegression().fit(pf.fit_transform(Xtr), ytr)
    ax.plot(grid["x"], m.predict(pf.transform(grid)), label=f"degree {degree}")
ax.set_ylim(-1, 6.5)
ax.set_xlabel("income (scaled 0–1)")
ax.set_ylabel("price ($100k)")
ax.legend(fontsize=8)
plt.show()

In [ ]:
from sklearn.metrics import mean_absolute_error

print("degree   train MAE   test MAE")
for degree in (1, 2, 4, 8, 15):
    pf = PolynomialFeatures(degree=degree, include_bias=False)
    A, B = pf.fit_transform(Xtr), pf.transform(Xte)
    m = LinearRegression().fit(A, ytr)
    train_err = mean_absolute_error(ytr, m.predict(A))
    test_err = mean_absolute_error(yte, m.predict(B))
    print(f"  {degree:2}       {train_err:.3f}     {test_err:.3f}")

In [ ]:
pf15 = PolynomialFeatures(degree=15, include_bias=False)
m15 = LinearRegression().fit(pf15.fit_transform(Xtr), ytr)

biggest = max(abs(c) for c in m15.coef_)
print(f"degree-1 largest |weight|:  6")
print(f"degree-15 largest |weight|: {biggest:,.0f}")

In [ ]:
from sklearn.linear_model import Ridge

A15, B15 = pf15.fit_transform(Xtr), pf15.transform(Xte)
print("alpha     train MAE   test MAE   largest |weight|")
for alpha in (1e-6, 0.001, 1.0, 100.0):
    r = Ridge(alpha=alpha).fit(A15, ytr)
    train_err = mean_absolute_error(ytr, r.predict(A15))
    test_err = mean_absolute_error(yte, r.predict(B15))
    big = max(abs(c) for c in r.coef_)
    print(f"{alpha:<8}  {train_err:.3f}      {test_err:.3f}      {big:,.2f}")

In [ ]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.01, max_iter=50000).fit(A15, ytr)
kept = sum(1 for c in lasso.coef_ if c != 0)
print(f"features kept: {kept} of 15")
print(f"test MAE: {mean_absolute_error(yte, lasso.predict(B15)):.3f}")

In [ ]:
model = Ridge(alpha=1.0)
model.fit(A15, ytr)

run_tests([
    ("honest test MAE", round(
        mean_absolute_error(yte, model.predict(B15)), 3), 0.685),
    ("weights stayed calm", round(max(abs(c) for c in model.coef_), 3), 1.911),
])

In [ ]:
def l2_penalty(weights, alpha):
    return alpha * sum(w * w for w in weights)

def ridge_cost(y_true, y_pred, weights, alpha):
    gaps = [(t - p) ** 2 for t, p in zip(y_true, y_pred)]
    return sum(gaps) / len(gaps) + l2_penalty(weights, alpha)

run_tests([
    ("rent on two weights", l2_penalty([0.3, 0.4], 0.5), 0.125),
    ("no rent at alpha zero", l2_penalty([5.0, -5.0], 0.0), 0.0),
    ("big weights cost more", l2_penalty([10.0], 1.0), 100.0),
    ("cost = mse + penalty",
     ridge_cost([2.0, 3.0], [1.0, 3.0], [0.3, 0.4], 0.5), 0.625),
    ("penalty ignores fit quality",
     ridge_cost([1.0, 1.0], [1.0, 1.0], [2.0], 1.0), 4.0),
], tol=1e-9)